In [1]:
import pandas as pd

df = pd.read_csv('data\data.csv')

#  Standardize Team Names
team_mapping = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru'
}

# Apply the mapping to all columns that contain team names
df['team1'] = df['team1'].replace(team_mapping)
df['team2'] = df['team2'].replace(team_mapping)
df['winner'] = df['winner'].replace(team_mapping)
df['toss_winner'] = df['toss_winner'].replace(team_mapping)

#  Handle Missing Values

print("Missing values before cleaning:\n", df.isnull().sum())

df.dropna(subset=['winner'], inplace=True) 

print(f"\nShape of data after cleaning: {df.shape}")
df.head()

<>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\gunja\AppData\Local\Temp\ipykernel_23068\3204957936.py:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  df = pd.read_csv('data\data.csv')
C:\Users\gunja\AppData\Local\Temp\ipykernel_23068\3204957936.py:3: DtypeWarning: Columns (0: season, 1: result_margin) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data\data.csv')


Missing values before cleaning:
 match_id                            0
inning                              0
batting_team                        0
bowling_team                        0
over                                0
ball                                0
batter                              0
bowler                              0
non_striker                         0
batsman_runs                        0
extra_runs                          0
total_runs                          0
extras_type                    246795
is_wicket                           0
player_dismissed                    0
dismissal_kind                      0
fielder                        251566
id                                  0
season                              0
city                                0
date                                0
match_type                          0
player_of_match                     0
venue                               0
team1                               0
team2            

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,...,method,umpire1,umpire2,is_legal,legal_ball,all_balls_over_ball,adjusted_over_ball,match_number_of_that_season,matches_in_that_season,match_number_in_total
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,...,Normal,Asad Rauf,RE Koertzen,True,1,0.1,0.1,1,58,1
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,...,Normal,Asad Rauf,RE Koertzen,True,2,0.2,0.2,1,58,1
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,...,Normal,Asad Rauf,RE Koertzen,False,2,0.3,0.2,1,58,1
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,...,Normal,Asad Rauf,RE Koertzen,True,3,0.4,0.3,1,58,1
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,...,Normal,Asad Rauf,RE Koertzen,True,4,0.5,0.4,1,58,1


In [2]:
print(df.columns.tolist())

['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder', 'id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2', 'is_legal', 'legal_ball', 'all_balls_over_ball', 'adjusted_over_ball', 'match_number_of_that_season', 'matches_in_that_season', 'match_number_in_total']


In [3]:
team_A = 'Chennai Super Kings'
team_B = 'Mumbai Indians'

mask = ((df['team1'] == team_A) & (df['team2'] == team_B)) | ((df['team1'] == team_B) & (df['team2'] == team_A))
rivalry_df = df[mask]

# Count the wins
wins = rivalry_df['winner'].value_counts()
print(f"Head-to-Head: {team_A} vs {team_B}")
print(wins)

Head-to-Head: Chennai Super Kings vs Mumbai Indians
winner
Mumbai Indians         4776
Chennai Super Kings    4150
Name: count, dtype: int64


In [4]:
match_df = df.drop_duplicates(subset=['match_id']).copy()

venue_name = 'Wankhede Stadium'
venue_df = match_df[match_df['venue'].str.contains(venue_name, na=False, case=False)]

print(f"Total MATCHES played at {venue_name}: {len(venue_df)}")

batting_first_wins = len(venue_df[venue_df['result'] == 'runs'])
chasing_wins = len(venue_df[venue_df['result'] == 'wickets'])

print(f"Won Batting First: {batting_first_wins}")
print(f"Won Chasing (Batting Second): {chasing_wins}")

Total MATCHES played at Wankhede Stadium: 118
Won Batting First: 53
Won Chasing (Batting Second): 64


In [5]:
import plotly.express as px

chart_data = pd.DataFrame({
    'Outcome': ['Batting First', 'Chasing'],
    'Wins': [batting_first_wins, chasing_wins]
})

fig = px.pie(chart_data, values='Wins', names='Outcome', 
             title=f'Toss Advantage at {venue_name} (2008-2024)',
             color='Outcome',
             color_discrete_map={'Batting First':'#1f77b4', 'Chasing':'#ff7f0e'})

fig.show()

In [6]:
print("Starting Feature Engineering for ML Model...")

chase_df = df[df['inning'] == 2].copy()

chase_df['current_score'] = chase_df.groupby('match_id')['total_runs'].cumsum()

chase_df['wickets_fallen'] = chase_df.groupby('match_id')['is_wicket'].cumsum()
chase_df['wickets_left'] = 10 - chase_df['wickets_fallen']

chase_df['balls_bowled'] = (chase_df['over'] * 6) + chase_df['ball']
chase_df['balls_left'] = 120 - chase_df['balls_bowled']

chase_df['balls_left'] = chase_df['balls_left'].apply(lambda x: x if x > 0 else 0)

print("✅ Features successfully engineered!")

chase_df[['match_id', 'batting_team', 'current_score', 'wickets_left', 'balls_left']].head()

Starting Feature Engineering for ML Model...
✅ Features successfully engineered!


,match_id,batting_team,current_score,wickets_left,balls_left
124,335982,Royal Challengers Bangalore,1,10,119
125,335982,Royal Challengers Bangalore,2,10,118
126,335982,Royal Challengers Bangalore,2,10,117
127,335982,Royal Challengers Bangalore,3,10,116
128,335982,Royal Challengers Bangalore,4,10,115


In [7]:
import numpy as np

print("Calculating Advanced ML Features...")

inning1_df = df[df['inning'] == 1]
target_scores = inning1_df.groupby('match_id')['total_runs'].sum().reset_index()
target_scores.rename(columns={'total_runs': 'target'}, inplace=True)

chase_df = chase_df.merge(target_scores, on='match_id', how='left')

chase_df['runs_left'] = chase_df['target'] - chase_df['current_score']
chase_df['runs_left'] = chase_df['runs_left'].apply(lambda x: x if x > 0 else 0)

chase_df['crr'] = (chase_df['current_score'] * 6) / chase_df['balls_bowled']

chase_df['rrr'] = np.where(chase_df['balls_left'] > 0, (chase_df['runs_left'] * 6) / chase_df['balls_left'], 0)

def result_label(row):
    return 1 if row['batting_team'] == row['winner'] else 0
chase_df['result'] = chase_df.apply(result_label, axis=1)

final_ml_df = chase_df[['batting_team', 'bowling_team', 'venue', 'runs_left', 'balls_left', 'wickets_left', 'target', 'crr', 'rrr', 'result']]

final_ml_df = final_ml_df.dropna()

print(" Advanced Feature Engineering Complete!")
print("\n Preview of our Final ML Dataset:")
print(final_ml_df.head(3))

final_ml_df.to_csv('data/ml_data.csv', index=False)
print("\n Saved final dataset to 'data/ml_data.csv'")

Calculating Advanced ML Features...
 Advanced Feature Engineering Complete!

 Preview of our Final ML Dataset:
                  batting_team           bowling_team                  venue  \
0  Royal Challengers Bangalore  Kolkata Knight Riders  M Chinnaswamy Stadium   
1  Royal Challengers Bangalore  Kolkata Knight Riders  M Chinnaswamy Stadium   
2  Royal Challengers Bangalore  Kolkata Knight Riders  M Chinnaswamy Stadium   

   runs_left  balls_left  wickets_left  target  crr        rrr  result  
0        221         119            10     222  6.0  11.142857       0  
1        220         118            10     222  6.0  11.186441       0  
2        220         117            10     222  4.0  11.282051       0  

 Saved final dataset to 'data/ml_data.csv'


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import pickle

print(" Loading dataset and preparing Machine Learning pipeline...")

# 1. Load the processed ML data
ml_df = pd.read_csv('data/ml_data.csv')

# 2. Separate features (X) and target variable (y)
X = ml_df[['batting_team', 'bowling_team', 'venue', 'runs_left', 'balls_left', 'wickets_left', 'target', 'crr', 'rrr']]
y = ml_df['result']

# 3. Split data into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Set up One-Hot Encoding for categorical text columns
# This converts text names of teams and venues into binary numbers the model can understand
cf = ColumnTransformer([
    ('trf', OneHotEncoder(drop='first', sparse_output=False), ['batting_team', 'bowling_team', 'venue'])
], remainder='passthrough')

# 5. Create a Scikit-Learn Pipeline
# The pipeline links the data encoder and the Logistic Regression model together
pipe = Pipeline(steps=[
    ('step1', cf),
    ('step2', LogisticRegression(solver='liblinear'))
])

# 6. Train the model
print(" Training the Logistic Regression model...")
pipe.fit(X_train, y_train)

# 7. Evaluate accuracy
y_pred = pipe.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f" Model Training Complete! Accuracy: {accuracy * 100:.2f}%")

# 8. Save the trained pipeline as a pickle file
with open('model.pkl', 'wb') as f:
    pickle.dump(pipe, f)
print(" Model successfully exported and saved as 'model.pkl'")

 Loading dataset and preparing Machine Learning pipeline...
 Training the Logistic Regression model...
 Model Training Complete! Accuracy: 87.76%
 Model successfully exported and saved as 'model.pkl'


In [10]:
# Let's mock a fake match situation to see what the model predicts
# Target 180, 45 runs left, 30 balls left, 7 wickets left, CRR 9.0, RRR 9.0
sample_input = pd.DataFrame([{
    'batting_team': 'Chennai Super Kings',
    'bowling_team': 'Mumbai Indians',
    'venue': 'Wankhede Stadium',
    'runs_left': 45,
    'balls_left': 30,
    'wickets_left': 7,
    'target': 180,
    'crr': 9.0,
    'rrr': 9.0
}])

# predict_proba returns [Probability of Loss, Probability of Win]
probabilities = pipe.predict_proba(sample_input)
print(f"Probability of Chasing Team Winning: {probabilities[0][1] * 100:.2f}%")

Probability of Chasing Team Winning: 83.45%
